# **Phân tích dữ liệu khám phá (EDA) bằng Python**

## **Giới thiệu**

Phân tích khám phá dữ liệu được thực hiện trên bộ dữ liệu theo dõi bệnh tiểu đường, bao gồm các thông tin về thời gian, loại chỉ số (Code), và giá trị đo lường (Value). Dữ liệu được thu thập từ nhiều file nhỏ, mỗi file chứa các bản ghi theo định dạng chuỗi thời gian.

### **Nguồn dữ liệu**

Thiết bị điện tử: ghi dữ liệu với dấu thời gian thực tế (timestamp).

Hồ sơ giấy: chỉ có các mốc thời gian logic (ví dụ: ăn sáng, ăn trưa, ăn tối, đi ngủ).

Gán giờ cố định:
- Ăn sáng → 08:00
- Ăn trưa → 12:00
- Ăn tối → 18:00
- Đi ngủ → 22:00
    
=> Như vậy, hồ sơ giấy sử dụng thời gian giả định, còn thiết bị điện tử cung cấp thời gian thực tế.

### **Cấu trúc tập tin**

Mỗi bản ghi có 4 trường, được phân cách bằng dấu tab, mỗi dòng là một bản ghi:
- Date (Ngày) – ngày ghi nhận; định dạng MM-DD-YYYY
- Time (Giờ) – giờ ghi nhận; định dạng HH:MM
- Code (Mã) – mã loại chỉ số 
- Value (Giá trị) – giá trị đo lường

### **Phân tích dữ liệu ban đầu**

Cách thực hiện: Sử dụng các thư viện mạnh mẽ của Python

In [2]:
# import thư viện pandas, và các module cần thiết khác
import pandas as pd
import glob
import os
from pathlib import Path

### **Định nghĩa mã**

| Code | Ý nghĩa |
|------|---------|
| 33   | Liều insulin Regular |
| 34   | Liều insulin NPH |
| 35   | Liều insulin UltraLente |
| 48   | Đo glucose (không xác định) |
| 57   | Đo glucose (không xác định) |
| 58   | Đo glucose trước bữa sáng |
| 59   | Đo glucose sau bữa sáng |
| 60   | Đo glucose trước bữa trưa |
| 61   | Đo glucose sau bữa trưa |
| 62   | Đo glucose trước bữa tối |
| 63   | Đo glucose sau bữa tối |
| 64   | Đo glucose trước bữa phụ (snack) |
| 65   | Triệu chứng hạ đường huyết |
| 66   | Ăn uống bình thường |
| 67   | Ăn nhiều hơn bình thường |
| 68   | Ăn ít hơn bình thường |
| 69   | Vận động bình thường |
| 70   | Vận động nhiều hơn bình thường |
| 71   | Vận động ít hơn bình thường |
| 72   | Sự kiện đặc biệt (không xác định) |

_____

## **Load the dataset**

In [3]:
# Đường dẫn tới thư mục chứa các file
path = r"D:\Github\Data_Analysis_SGU_2025\Data_analysis\practice\lab3\file\diabetes-data\diabetes-data"

# Lấy tất cả các file bắt đầu bằng 'data-' và loại bỏ 'Data-Codes'
files = glob.glob(path + r"\data-*")
files = [f for f in files if os.path.basename(f).lower() != "data-codes"]

df_list = []
for file in files:
    try:
        df = pd.read_csv(file, sep="\t", header=None, names=["Date", "Time", "Code", "Value"])
        df_list.append(df)
    except Exception as e:
        print(f"Lỗi khi đọc file: {file}")
        print(e)

# Gộp lại nếu có ít nhất một file đọc thành công
if df_list:
    df = pd.concat(df_list, ignore_index=True)
    print("Đọc dữ liệu thành công! Số dòng:", len(df))
else:
    print("Không có file nào được đọc thành công.")

Đọc dữ liệu thành công! Số dòng: 29330


In [11]:
df

,Date,Time,Code,Value
0,04-21-1991,9:09,58,100
1,04-21-1991,9:09,33,9
2,04-21-1991,9:09,34,13
3,04-21-1991,17:08,62,119
4,04-21-1991,17:08,33,7
...,...,...,...,...
29325,05-09-1989,08:00,33,1.0
29326,05-09-1989,08:00,34,7.0
29327,05-10-1989,08:00,34,7.0
29328,05-11-1989,08:00,34,7.0


In [19]:
# # Kiểm tra kích thước của DataFrame: số dòng và số cột
df.shape

(29330, 4)

In [20]:
# Hiển thị 5 dòng đầu tiên của DataFrame
df.head()

,Date,Time,Code,Value
0,04-21-1991,9:09,58,100
1,04-21-1991,9:09,33,9
2,04-21-1991,9:09,34,13
3,04-21-1991,17:08,62,119
4,04-21-1991,17:08,33,7


In [21]:
# Hiển thị 10 dòng đầu tiên của DataFrame
df.head(10)

,Date,Time,Code,Value
0,04-21-1991,9:09,58,100
1,04-21-1991,9:09,33,9
2,04-21-1991,9:09,34,13
3,04-21-1991,17:08,62,119
4,04-21-1991,17:08,33,7
5,04-21-1991,22:51,48,123
6,04-22-1991,7:35,58,216
7,04-22-1991,7:35,33,10
8,04-22-1991,7:35,34,13
9,04-22-1991,13:40,33,2


**Các loại biến**

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29330 entries, 0 to 29329
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    29297 non-null  object
 1   Time    29330 non-null  object
 2   Code    29330 non-null  int64 
 3   Value   29297 non-null  object
dtypes: int64(1), object(3)
memory usage: 916.7+ KB


**Kiểm tra các giá trị null/NA**

In [17]:
df.isnull().sum()

Date     33
Time      0
Code      0
Value    33
dtype: int64

**Các thống kê của các mã**

In [25]:
# Hiển thị thống kê mô tả cho các cột số trong DataFrame
df.describe()

,Code
count,29330.000000
mean,46.428606
std,13.453219
min,0.000000
25%,33.000000
50%,48.000000
75%,60.000000
max,72.000000


Kết luận về các thống kê của các mã: 

- Tổng số mã ghi nhận (count): 29,330 dòng dữ liệu.

- Giá trị trung bình (mean) của mã Code: khoảng 46.43, cho thấy phần lớn các mã nằm trong nhóm theo dõi glucose và insulin.

- Độ lệch chuẩn (std): khoảng 13.45, phản ánh mức độ phân tán của các mã quanh giá trị trung bình.

- Giá trị nhỏ nhất (min): 0 => có thể là mã đặc biệt hoặc lỗi nhập liệu.

- Phân vị thứ nhất (25%): 33.0

- Trung vị (50%): 48.0

- Phân vị thứ ba (75%): 60.0 => Các phân vị này cho thấy phần lớn dữ liệu tập trung quanh các mã phổ biến như insulin (33–35) và đo glucose (58–64).

- Giá trị lớn nhất (max): 72 => nằm trong nhóm mã hành vi hoặc sự kiện đặc biệt.

**Phân bố các mã Code trong dữ liệu**

In [23]:
df['Code'].value_counts()

Code
33    9518
34    3830
58    3518
62    3160
60    2771
48    1883
35    1053
57     990
64     904
65     331
67     326
63     219
66     154
70     139
56     119
71      98
72      94
69      68
61      66
68      34
0       33
59      20
4        1
36       1
Name: count, dtype: int64

Kết luận về kết quả phân bố các mã trong dữ liệu:

- Mã 33 (Liều insulin Regular) xuất hiện nhiều nhất, cho thấy đây là loại insulin được sử dụng phổ biến trong quá trình điều trị.

- Các mã đo glucose như 62 (sau bữa trưa), 58 (không xác định), 64 (sau bữa tối), và 60 (sau bữa ăn) cũng có tần suất cao, phản ánh việc theo dõi đường huyết thường xuyên theo thời điểm trong ngày.

- Các mã hành vi và sự kiện như 71 (triệu chứng hạ đường huyết), 72 (ăn uống bình thường), và 76 (vận động nhiều hơn bình thường) xuất hiện ít hơn, cho thấy đây là các tình huống đặc biệt hoặc ít gặp.

## **EDA trực quan**

### **Biến động các chỉ số theo thời gian**

### **Thống kê tổng quan theo từng mã (Code)**

### **Xu hướng dài hạn qua trung bình trượt (Rolling Mean)**